# softmax回归的从零开始实现
---
## 环境配置


In [ ]:
%pip install pypto==0.2.0 torch torch_npu numpy

In [ ]:
import os, sys
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import pypto
import torch
import torch_npu
import numpy as np

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
mode = pypto.RunMode.NPU

---

## 练习3.6.1

本节直接实现了基于数学定义softmax运算的`softmax`函数。这可能会导致什么问题？提示：尝试计算$\exp(50)$的大小。


### 解答


以下使用 `torch` 编程进行验证：


In [ ]:
import numpy as np

def softmax(X):
    X_exp = np.exp(X)
    partition = X_exp.sum(1, keepdims=True)
    return X_exp / partition

try:
    softmax(np.array([[50]]))
except Exception as e:
    print(e)

使用 `PyPTO` 编程进行验证:


In [11]:
@pypto.frontend.jit()
def softmax_naive_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(8, 8)
    x_exp = pypto.exp(x)
    partition = pypto.sum(x_exp, dim=-1, keepdim=True)
    out[:] = x_exp / partition

def softmax(X):
    out = torch.empty_like(X)
    softmax_naive_kernel(X.contiguous(), out)
    return out

In [12]:
try:
    print(softmax(torch.tensor([[500.0, 1.0]], device='npu:0')))
except Exception as e:
    print(e)

tensor([[nan, 0.]], device='npu:0')


由于指数函数的值域是 $(0,\infty)$，因此可能会出现数值上溢的问题。这就是说，由于 $\exp(500)$ 的结果非常大，它可能超出计算机所能表示的范围，从而被近似为无穷大（inf）。这会带来一些问题，例如在反向传播时可能会出现NaN（不是数字）的情况。解决这个问题的一种常用技巧是，在计算softmax之前，先从所有输入中减去输入中的最大值。这样可以确保指数函数的输入不会太大而导致数值上溢。


---


## 练习3.6.2

本节中的函数`cross_entropy`是根据交叉熵损失函数的定义实现的。它可能有什么问题？提示：考虑对数的定义域。


### 解答


交叉熵损失函数定义了log函数。当模型预测概率为0时，log函数的值为负无穷。因此，在实际计算时，通常忽略预测概率接近0的样本对损失函数的贡献。这可能会导致模型欠拟合，并且在训练过程中难以收敛。


---


## 练习3.6.3

请提出一个解决方案来解决上述两个问题。


### 解答


对于 PyPTO，解决这两个问题的关键是使用数值稳定的实现方式。

对于 softmax，不应直接计算 `exp(X)`，而应先减去每行最大值，再计算
`exp(X - max(X)) / sum(exp(X - max(X)))`。PyPTO 内置的 `pypto.softmax` 已经采用了稳定实现，因此优先直接使用该算子。

对于交叉熵，不应直接对未经保护的概率做 `log`。一种简单方法是先对正确类别概率做下界截断，例如 `clip(p, min=1e-12)`，再计算 `-log(p)`；更稳定的方法是直接从 logits 计算交叉熵，将 softmax 与 log 合并处理，从而同时避免指数上溢和对数输入为 0 的问题。


---


## 练习3.6.4

返回概率最大的分类标签总是最优解吗？例如，医疗诊断场景下可以这样做吗？


### 解答


在某些情况下，返回概率最大的分类标签可能不是最优解。例如，在医疗诊断场景下，我们更关心误诊率和漏诊率等错误类型之间的权衡，并且尽可能避免小概率事件的发生。


---


## 练习3.6.5

假设我们使用softmax回归来预测下一个单词，可选取的单词数过多可能会带来哪些问题?


### 解答


根据题意可知，如果可选取单词数量过多，有以下问题：
1. 单词量过大，会导致计算的复杂度增加
2. 需要计算更多模型参数，并会导致模型复杂度增加
3. 所有的单词所得概率容易接近0，导致难以判断输出结果
4. 在训练期间需要处理更多数据，并且预测时间也会变得更长


---
## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)
